# 03 · Cell type annotation — GSE216146

Assigns a cell type to every Leiden cluster (`annotation.cluster_key`, resolution 0.5) using three independent sources:
1. **Marker genes** (`annotation.marker_genes`) and data-driven cluster markers (Wilcoxon).
2. **CellTypist** with the `Mouse_Whole_Brain` model (adult mouse brain, Allen atlas subclasses), mapped to broad
   classes with `annotation.celltypist_classes`.
3. The **authors' labels** (`MajorCellType`) as an external cross-check.

The final labels are written in `config.yaml` (`annotation.cluster_labels`); this notebook asserts that each cluster's
majority CellTypist class matches the expected class, so a change in the clustering is caught instead of silently
mislabelling cells.

**Input:** `results/02_integrated.h5ad` · **Output:** `results/03_annotated.h5ad` with `obs["cell_type"]`,
`obs["cell_type_fine"]` and the CellTypist columns (`ct_*`); plots in `results/plots/03_*`, tables in `results/tables/03_*`.

Notes (checked on 2026-11-13):
- The tissue looks hippocampal plus choroid plexus: dentate gyrus granule neurons, DG neuroblasts and Cajal-Retzius cells.
- Cluster 5 (~100 cells) has microglia markers at low detection, few genes (median ~600) and low CellTypist confidence:
  it is labelled **Low-quality microglia** and kept here; downstream notebooks decide whether to exclude it.
- *Ttr* (choroid plexus) is detected in ~55 % of all cells of every type: ambient RNA. See section 5.

In [ ]:
import gc
import sys
from importlib.metadata import version
from pathlib import Path

import celltypist
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "scripts"))
import sc_utils as scu

cfg = scu.load_config(ROOT / "config.yaml")
p = cfg["annotation"]
ckey = p["cluster_key"]
scu.setup_plotting()
print("scanpy", version("scanpy"), "| celltypist", version("celltypist"))
print("model:", p["celltypist_model"], "| cluster key:", ckey)

In [ ]:
adata = sc.read_h5ad(cfg["paths"]["results_dir"] / "02_integrated.h5ad")
assert ckey in adata.obs and p["over_clustering"] in adata.obs
adata

## 1. Marker genes per cluster

In [ ]:
markers = {ct: [g for g in genes if g in adata.var_names] for ct, genes in p["marker_genes"].items()}
missing = sorted(set(sum(p["marker_genes"].values(), [])) - set(sum(markers.values(), [])))
print("markers not in the data:", missing or "none")

fig = sc.pl.dotplot(adata, markers, groupby=ckey, standard_scale="var", return_fig=True)
fig.savefig(cfg["paths"]["plots_dir"] / "03_dotplot_markers_by_cluster.png", dpi=300, bbox_inches="tight")
fig.show()

In [ ]:
# Data-driven markers: top genes per cluster (Wilcoxon on log-normalized X, one cluster vs rest)
sc.tl.rank_genes_groups(adata, groupby=ckey, method="wilcoxon", pts=True, key_added="rank_cluster")
top = sc.get.rank_genes_groups_df(adata, group=None, key="rank_cluster")
top = (top.query("logfoldchanges > 1 and pvals_adj < 0.05")
          .sort_values(["group", "scores"], ascending=[True, False])
          .groupby("group", observed=True).head(10))
top.to_csv(cfg["paths"]["tables_dir"] / "03_cluster_top_markers.tsv", sep="\t", index=False)
top.groupby("group", observed=True)["names"].agg(", ".join).to_frame("top 10 markers")

## 2. CellTypist (`Mouse_Whole_Brain`)

`X` is already log1p of counts normalized to 10,000, which is what CellTypist expects. Majority voting runs over
the fine Leiden clusters (`annotation.over_clustering`). The ~330 Allen subclasses are mapped to broad classes.

In [ ]:
celltypist.models.download_models(model=[p["celltypist_model"]])
pred = celltypist.annotate(adata, model=p["celltypist_model"], majority_voting=p["majority_voting"],
                           over_clustering=p["over_clustering"])
adata.obs["ct_label"] = pred.predicted_labels["predicted_labels"].astype("category")
adata.obs["ct_majority"] = pred.predicted_labels["majority_voting"].astype("category")
adata.obs["ct_conf"] = pred.probability_matrix.max(axis=1).astype(np.float32)
adata.obs["ct_class"] = scu.broad_class(adata.obs["ct_label"], p["celltypist_classes"])
adata.obs["ct_majority_class"] = scu.broad_class(adata.obs["ct_majority"], p["celltypist_classes"])
del pred
print("unmapped labels:", sorted(adata.obs.loc[adata.obs["ct_class"] == "Unmapped", "ct_label"].unique()) or "none")
adata.obs["ct_class"].value_counts()

In [ ]:
fig = sc.pl.umap(adata, color=["ct_class", "ct_conf", "MajorCellType"], ncols=3, wspace=0.4, size=8,
                 return_fig=True)
scu.save_fig(fig, cfg, "03_umap_celltypist")
plt.show()

## 3. Cross-check per cluster

For each cluster: majority CellTypist class (per cell), median confidence, majority author label and the label from
`config.yaml`. The assertion fails if a cluster's majority CellTypist class differs from the expected one.

In [ ]:
obs = adata.obs
labels = pd.DataFrame(p["cluster_labels"]).T
labels.index = labels.index.astype(str)
assert set(labels.index) == set(obs[ckey].cat.categories), "cluster_labels must cover every cluster exactly"

check = pd.concat([
    obs.groupby(ckey, observed=True).size().rename("cells"),
    scu.majority(obs, ckey, "ct_class"),
    obs.groupby(ckey, observed=True)["ct_conf"].median().rename("ct_conf_median"),
    scu.majority(obs, ckey, "MajorCellType"),
    obs.groupby(ckey, observed=True)["n_genes_by_counts"].median().rename("median_genes"),
    labels,
], axis=1)
check["agree"] = check["ct_class_majority"].astype(str) == check["celltypist"].astype(str)
check.to_csv(cfg["paths"]["tables_dir"] / "03_cluster_annotation_check.tsv", sep="\t")
assert check["agree"].all(), check.loc[~check["agree"]]
check.round(2)

## 4. Final labels

In [ ]:
adata.obs["cell_type"] = obs[ckey].astype(str).map(labels["cell_type"]).astype("category")
adata.obs["cell_type_fine"] = obs[ckey].astype(str).map(labels["fine"]).astype("category")
assert adata.obs["cell_type"].notna().all()

fig = sc.pl.umap(adata, color=["cell_type", "cell_type_fine"], ncols=2, wspace=0.6, size=8, return_fig=True)
scu.save_fig(fig, cfg, "03_umap_cell_type")
plt.show()

In [ ]:
fig = sc.pl.dotplot(adata, markers, groupby="cell_type", standard_scale="var", return_fig=True)
fig.savefig(cfg["paths"]["plots_dir"] / "03_dotplot_markers_by_cell_type.png", dpi=300, bbox_inches="tight")
fig.show()

In [ ]:
# Agreement with the authors' labels (row-normalized)
ct = pd.crosstab(adata.obs["cell_type"], adata.obs["MajorCellType"], normalize="index")
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(ct, cmap="Blues", vmin=0, vmax=1, annot=ct.round(2).where(ct >= 0.05), fmt="", ax=ax,
            cbar_kws={"label": "fraction of cells"})
ax.set_xlabel("author MajorCellType"); ax.set_ylabel("cell_type (this analysis)")
scu.save_fig(fig, cfg, "03_heatmap_vs_author_labels")
plt.show()

In [ ]:
# Cells per sample and cell type (input for 04_composition)
counts = pd.crosstab([adata.obs["condition"], adata.obs["biosample"]], adata.obs["cell_type"])
counts.to_csv(cfg["paths"]["tables_dir"] / "03_cell_type_counts.tsv", sep="\t")
counts

## 5. Ambient RNA check

Genes from abundant, highly expressing cells (choroid plexus *Ttr*, erythrocyte hemoglobins) leak into every droplet.
The counts are not ambient-corrected (no empty droplets are available), so these genes can show up as artefactual
DE hits. Fraction of cells with at least one count, per cell type and per sample:

In [ ]:
amb = [g for g in p["ambient_genes"] if g in adata.var_names]
detected = pd.DataFrame((adata[:, amb].layers["counts"] > 0).toarray(), columns=amb, index=adata.obs_names)
pd.concat({
    "by cell_type": detected.groupby(adata.obs["cell_type"], observed=True).mean(),
    "by biosample": detected.groupby(adata.obs["biosample"], observed=True).mean(),
}).round(2)

## 6. Save

In [ ]:
adata.uns.pop("rank_cluster", None)          # large; markers are saved as a table
out_h5ad = cfg["paths"]["results_dir"] / "03_annotated.h5ad"
adata.write_h5ad(out_h5ad, compression="gzip")
print("wrote", out_h5ad)
adata

In [ ]:
del adata
_ = gc.collect()